In [13]:
# 动手练习RNNcell
import torch
import torch.nn as nn
import torch.optim as optim

# 1.字符表
idx2char = ['e','h','l','o']

# 2.输入序列： hello
x_data  = [1,0,2,2,3]

# 3.目标序列： ohlol
y_data = [3,1,2,3,2]

# One-hot 查询表
one_hot_lookup = [
    [1,0,0,0], # 0 -> e
    [0,1,0,0], # 1 -> h
    [0,0,1,0], # 2 -> l
    [0,0,0,1]  # 3 -> o
]

# 将输入编号转换成0ne_hot
x_one_hot = [one_hot_lookup[x] for x in x_data]

# 基本参数
batch_size =1
seq_len =5
input_size =4
hidden_size =8
num_class = len(idx2char)

# 7.转化为PyTorch tensor
inputs = torch.tensor(x_one_hot,
                      dtype = torch.float32,
                      ).view(seq_len,batch_size,input_size)
labels = torch.tensor(y_data,
                      dtype = torch.long).view(seq_len,batch_size)

# 8.查看数据
print('输入字符',''.join(idx2char[x] for x in x_data ))
print('输出字符',''.join(idx2char[x] for x in y_data))

print("inputs.shape =", inputs.shape)
print("labels.shape =", labels.shape)

print("\ninputs：")
print(inputs)

print("\nlabels：")
print(labels)

# 先定义RNN模型

# seq_len是由外面的数据形状和循环控制的
class RNNLinearModel(torch.nn.Module):
    def __init__(self,input_size,hidden_size,num_layers,num_class):
        super().__init__()
        # 这两个参数通常在创建模型之后不会发生变化
        # 所以适合保存在模型内部
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        #创建RNN
        self.rnn = torch.nn.RNN(
            input_size=input_size,
            hidden_size = hidden_size,
            num_layers = num_layers,
            batch_first = False,
        )

        # 把隐藏状态转化为类别数
        # 加入Linear后Loss降得更低
        # RNN的8维记忆-> Linear -> 4个类别分数
        # 同时Linear也参与了优化器的优化
        self.fc = torch.nn.Linear(hidden_size,num_class)

    def forward(self,x,hidden):
        # 一次性处理完序列

        # outputs表示每个时间步的结果[hidden0,hidden1,hidden2....]
        # 它的形状是【seq_len,batch_size,hidden_size】
        outputs,final_hidden = self.rnn(x,hidden)
        # 每个时间步的隐藏状态都通过Linear
        logits = self.fc(outputs)
        return logits,final_hidden

    def init_hidden(self,batch_size):
        # nn.RNN 的初始hidden是三维的
        hidden = torch.zeros(self.num_layers,batch_size,self.hidden_size)
        return hidden

# 设置层数并创建模型

# 这里表示一层RNN
num_layer= 1
rnn_net = RNNLinearModel(
    input_size = input_size,
    hidden_size = hidden_size,
    num_layers = num_layer,
    num_class = num_class
)

print(rnn_net)

# 创建初始隐藏状态
# 为什么不将batch_size 提前保存在模型中
# 因为batch可能会发生变化
# 例如训练中batch_size = 64,到最后可能就剩17
# 所以在调用中传进去更灵活
hidden = rnn_net.init_hidden(batch_size)

# 开始整理形状
# 目前模型输出为【seq_len,batch_size,hidden_size】->[5,1,4]
# 但是CrossEntropyLoss最常见的格式为
# 预测 【样本数量，类别数量】
# 标签【样本数量】
# 所以把输出从【5，1，4】变成【5x1,1】
# 即从【seq_len,batch_size,hidden_size】 ->【seq_len x batch_size,hidden_size】
# 标签从【5，1】 -[5]

# 把 seq_len 和 batch 维度合并
outputs_2d = outputs.reshape(-1,num_class)

# 把标签展平
labels_1d = labels.reshape(-1)
print("整理前 outputs.shape：", outputs.shape)
print("整理后 outputs_2d.shape：", outputs_2d.shape)

print("整理前 labels.shape：", labels.shape)
print("整理后 labels_1d.shape：", labels_1d.shape)

print("\n整理后的 outputs：")
print(outputs_2d)

print("\n整理后的 labels：")
print(labels_1d)

# 开始正式训练RNN

# 多分类损失函数
criterion = nn.CrossEntropyLoss()

# 优化器
optimizer = optim.Adam(rnn_net.parameters(),lr = 0.05)

# 训练15轮
for epoch in range(15):

    # 1.清空上一轮梯度
    optimizer.zero_grad()

    # 2.初始化隐藏状态
    hidden = rnn_net.init_hidden(batch_size)

    # 3.将整条序列送入RNN
    outputs,final_hidden = rnn_net(inputs,hidden)

    # 4.整理模型输出形状
    # outputs 是经过Linear 后的类别数
    # shape : [seq_len,batch_size,num_class]
    outputs_2d = outputs.reshape(-1,num_class)

    # 5,整理标签输出形状
    labels_1d = labels.reshape(-1)

    # 6.一次计算5个时间步的平均损失
    # 注意这里算出来的就是平均损失了
    loss = criterion(outputs_2d,labels_1d)

    # 7.反向传播
    loss.backward()

    # 8.更新参数
    optimizer.step()

    # 9.找到每个时间步预测的类别
    predicted_indices = outputs_2d.argmax(dim=1)

    # 10.把编号转化为字符
    predicted_string = ''.join(idx2char[index] for index in predicted_indices)

    print(f'Epoch [{epoch+1}/15]'
          f'预测：{predicted_string}'
          f'平均损失：{loss.item():4f}')

输入字符 hello
输出字符 ohlol
inputs.shape = torch.Size([5, 1, 4])
labels.shape = torch.Size([5, 1])

inputs：
tensor([[[0., 1., 0., 0.]],

        [[1., 0., 0., 0.]],

        [[0., 0., 1., 0.]],

        [[0., 0., 1., 0.]],

        [[0., 0., 0., 1.]]])

labels：
tensor([[3],
        [1],
        [2],
        [3],
        [2]])
RNNLinearModel(
  (rnn): RNN(4, 8)
  (fc): Linear(in_features=8, out_features=4, bias=True)
)
整理前 outputs.shape： torch.Size([5, 1, 4])
整理后 outputs_2d.shape： torch.Size([5, 4])
整理前 labels.shape： torch.Size([5, 1])
整理后 labels_1d.shape： torch.Size([5])

整理后的 outputs：
tensor([[-0.9616, -0.8494, -0.5594,  0.7359],
        [-0.9665,  0.8659, -0.6806, -0.9609],
        [-0.8399, -0.1025,  0.8278,  0.7439],
        [-0.9651, -0.9593, -0.0533,  0.9151],
        [-0.9840, -0.7950,  0.4341, -0.8501]], grad_fn=<ViewBackward0>)

整理后的 labels：
tensor([3, 1, 2, 3, 2])
Epoch [1/15]预测：hohho平均损失：1.431681
Epoch [2/15]预测：ooooo平均损失：1.210300
Epoch [3/15]预测：ooool平均损失：1.048884
Epoch [4/15]预测：oo